# DiD: IRA & CHIPS Act and factory construction

Question: how much of the boom in computer, electronic and electrical factory construction is caused by the August 2022 laws?
We compare this treated sector with the 6 other manufacturing subsectors, which serve as control sectors, using log real spending (2019 \$M). The computations are in [`src/did.py`](../src/did.py).

In [1]:
import sys

import polars as pl

sys.path.append('../src')
import data_processing as dp
import did
from plot_functions import set_theme, plot_event_study, plot_placebos, plot_specs

set_theme()
PANEL = dp.load_processed()['did_panel_monthly']
MFG = PANEL.filter(pl.col('sector').str.starts_with('mfg_'))

## 1. Design

![DiD causal graph](../reports/figures/dag_did.png)

The double difference removes stable differences between sectors (α in the graph, such as size or capital intensity) and shocks common to all sectors (λ, such as Fed rates or materials prices).
It does not remove a factor that evolves differently across sectors (U), such as the 2021 chip shortage or AI-driven demand.
Identification rests on three assumptions. Consistency requires a well-defined treatment, here the CHIPS + IRA package. No anticipation assumes that nothing reacts before the laws. Parallel trends assume that, without the laws, the treated sector would have evolved like the control sectors.

## 2. 2×2 DiD

In [2]:
did.cell_means(MFG).with_columns(pl.exclude('group').round(3))

group,before,after,change
str,f64,f64,f64
"""Treated""",9.036,11.043,2.007
"""Controls""",8.273,8.571,0.298
"""Gap treated − controls""",0.763,2.472,1.709


The before period runs from July 2019 to June 2021, the after period from August 2022 to August 2026, and the 13 transition months in between are excluded (section 3). The treated sector gains 2.007 in logs and the control sectors 0.298, hence δ = **1.709**.
The regression with sector and month fixed effects (TWFE) on the same months, $Y_{st} = \alpha_s + \lambda_t + \delta\, T_{st} + \varepsilon_{st}$, gives the same δ.

In [3]:
did.twfe(MFG)

{'delta': 1.7087749555144292, 'se_cluster': 0.09395508484994615}

The regression recovers δ = 1.709. Its sector-clustered standard error (0.094) is not reliable with a single treated sector, so uncertainty is assessed differently in section 5.

## 3. Pre-trends

In [4]:
es = did.event_study(MFG)
plot_event_study(es, did.BEFORE, did.TRANSITION,
                 title='Gap between the treated sector and control sectors, by quarter',
                 subtitle="Log gap, 0 = mean gap from July 2019 to June 2021")

Before the laws, the treated sector first catches up with the control sectors from 2017 to 2019 (−2.05 in 2017-Q1). The gap then stabilises from mid-2019 to mid-2021, between −0.15 and +0.13, and rises from 2021-Q4 (**+0.60**).
We use this plateau as the before period and exclude the transition, which may reflect anticipation as well as the chip shortage. A window starting in 2015 would mix the catch-up with the effect.
The plateau was chosen after looking at the event study, and section 6 shows the result with the window from 2015.

## 4. Estimation

In [5]:
did.readings(MFG).with_columns(pl.col('delta', 'effect').round(3))

reading,before,delta,effect
str,str,f64,f64
"""Upper bound: anticipation""","""2019-07 → 2021-06""",1.709,4.522
"""Lower bound: chip shortage""","""2021-07 → 2022-07""",1.061,1.89


The upper bound assumes that, without the laws, the sector would have stayed at the 2019-2021 plateau level. All of the late-2021 rise is then attributed to anticipation, and the effect is **+452%**.
The lower bound assumes that the transition rise, linked to the chip shortage, would have happened without the laws. The transition then serves as the reference and the effect falls to **+189%**.
The data cannot decide between the two.

In [6]:
did.dollars(MFG).with_columns(pl.col('effect').round(3), pl.col('bn_per_year').round(1))

bound,effect,bn_per_year
str,f64,f64
"""Upper bound: anticipation""",4.522,53.9
"""Lower bound: chip shortage""",1.89,43.0


In 2019 dollars, the laws would thus add \$43.0bn to \$53.9bn per year to the sector's factory construction spending since August 2022.
The gap between the bounds is smaller in dollars than in percent, because both start from the same observed spending and only the counterfactual share changes.

In [7]:
did.dynamics(es).with_columns(pl.col('coef', 'effect').round(3))

moment,quarter,coef,effect
str,str,f64,f64
"""First quarter""","""2022-Q3""",1.26,2.527
"""Peak""","""2023-Q4""",2.066,6.894
"""Last quarter""","""2026-Q3""",1.167,2.214


The effect goes from +253% in 2022-Q3 to a peak of +689% in 2023-Q4, then falls back to +221% in 2026-Q3 (July-August). For a flow of construction spending, this decline corresponds to the end of the construction wave, as plants are delivered.
The peak and the decline are measured against the plateau and should therefore be read as the upper bound.

## 5. Inference

In [8]:
plot_placebos(did.time_placebos(MFG), did.did_2x2(MFG),
              title='True δ and placebo DiDs at fake dates',
              subtitle='Same 2×2 DiD (24 months before, 13 excluded, 49 after) on fake dates')

None of the 53 placebos reaches the true δ, and the largest is **0.66**. Fake dates stop in June 2017 so that the after window does not overlap the 2021 rise.
Shifted by one month, with 24- and 49-month windows that overlap almost entirely, these fake dates are not independent draws. The 0 / 53 count is therefore not a p-value; what matters is the distance between the true δ and the largest placebo.
This distance falls from 1.05 (δ = 1.709) to 0.40 with the lower bound (δ = 1.061). The true δ stays above, but with a much smaller margin.

In [9]:
did.permutation(MFG).with_columns(pl.all().round(3))

delta_true,placebo_min,placebo_max,p
f64,f64,f64,f64
1.709,-0.677,-0.089,0.143


The cross-sector permutation test assigns the treatment to each of the 7 sectors in turn. The true δ (1.709) is the only positive one, with placebos ranging from −0.677 to −0.089.
The p-value is **0.143**, i.e. 1/7, the minimum possible with 7 sectors.

## 6. Robustness

In [10]:
plot_specs(did.robustness(PANEL), 'specification', 'effect', 'Main (transition excluded)',
           title='Estimated effect by specification', subtitle='Average effect since August 2022, exp(δ) − 1',
           xtitle='Estimated effect')

The effect stays between **+430% and +495%** without transportation equipment, which may contain EV plants, or with offices and commercial among the control sectors.
In levels, the effect reaches +591%, but this scale assumes that the treated sector would have gained as many dollars as the control sectors rather than the same percentage. We keep logs, since the question is about a percentage.
The window from 2015 gives **+915%** because it mixes the 2017-2019 catch-up with the effect.

## 7. Takeaways

- **Effect**: between +189% (lower bound, chip shortage) and +452% (upper bound, anticipation) since August 2022, i.e. \$43.0bn to \$53.9bn (2019 dollars) per year.
- **Dynamics**: peak of +689% in 2023-Q4, then +221% in mid-2026, measured from the plateau and thus read as the upper bound.
- **Pre-trends**: plateau from mid-2019 to mid-2021, chosen from the event study, then a rise from late 2021. The data do not separate anticipation from the chip shortage, hence the range.
- **Uncertainty**: for both bounds, the true δ exceeds the largest in-time placebo (0.66); across sectors, p = 1/7, the minimum possible.
- **Robustness**: stable across control sectors (+430% to +495%), very sensitive to the before window (+915% from 2015).
- **Limitation**: a single treated sector and a simultaneous AI boom, which the DiD cannot separate from the laws.